In [1]:
# ============================================================
# Cell 1 — Load maneuver data and inspect structure
# ============================================================
import numpy as np
import pandas as pd
from pathlib import Path

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 260)

FILES = {
    1: "robot_1_maneuver.csv",
    2: "robot_2_maneuver.csv",
    3: "robot_3_maneuver.csv",
    4: "robot_4_maneuver.csv",
}

data = {}
print("=" * 100)
print(" MANEUVER DATA LOAD")
print("=" * 100)

for rb, fname in FILES.items():
    p = Path(fname)
    if not p.exists():
        print(f"  MISSING: {fname}")
        continue
    d = pd.read_csv(p)
    data[rb] = d
    dur = (d["t_ms"].iloc[-1] - d["t_ms"].iloc[0]) / 1000.0
    print(f"  R{rb}: {len(d):>4d} rows, {dur:>6.2f} s, "
          f"dt ≈ {np.median(np.diff(d['t_ms']))/1000:.3f} s")

# ---------- Column overview ----------
print("\n" + "=" * 100)
print(" COLUMN INVENTORY")
print("=" * 100)
ref = data[1]
for i, col in enumerate(ref.columns):
    print(f"  {i:>2d}. {col:<20s}  "
          f"min={ref[col].min():>+10.4f}  "
          f"max={ref[col].max():>+10.4f}  "
          f"dtype={ref[col].dtype}")

# ---------- UWB peer distribution ----------
print("\n" + "=" * 100)
print(" UWB PEER DISTRIBUTION PER ROBOT")
print("=" * 100)
print(" Each row contains ONE UWB measurement. Which peer does it come from?")
print(f"\n{'robot':>5s} | " +
      " | ".join(f"peer {p}" for p in [1, 2, 3, 4]))
print("-" * 50)
for rb, d in data.items():
    counts = d["uwb_peer_id"].value_counts().to_dict()
    row = f"{rb:>5d} | "
    cells = []
    for p in [1, 2, 3, 4]:
        cells.append(f"{counts.get(p, 0):>6d}")
    print(row + " | ".join(cells))

# ---------- Initial and final positions ----------
print("\n" + "=" * 100)
print(" INITIAL AND FINAL POSITIONS")
print("=" * 100)
print(" Expected: square side 2.0 m → side 3.0 m, rotated 90° CCW.")
print(f"\n{'robot':>5s} | {'x_init':>8s} {'y_init':>8s} | "
      f"{'x_final':>8s} {'y_final':>8s} | "
      f"{'heading_init':>13s} {'heading_final':>13s}")
print("-" * 82)
for rb, d in data.items():
    first = d.iloc[0]
    last  = d.iloc[-1]
    print(f"{rb:>5d} | {first['x']:>+8.4f} {first['y']:>+8.4f} | "
          f"{last['x']:>+8.4f} {last['y']:>+8.4f} | "
          f"{np.degrees(first['heading']):>+12.2f}° "
          f"{np.degrees(last['heading']):>+12.2f}°")

# ---------- Actual vs nominal trajectory ----------
print("\n" + "=" * 100)
print(" TRAJECTORY METRICS")
print("=" * 100)
print(f"{'robot':>5s} | {'total path':>11s} | {'displacement':>12s} | "
      f"{'peak v_cmd':>11s} | {'peak omega':>11s}")
print("-" * 62)
for rb, d in data.items():
    dx = np.diff(d["x"].values)
    dy = np.diff(d["y"].values)
    path_len = float(np.sum(np.hypot(dx, dy)))
    disp = float(np.hypot(d["x"].iloc[-1] - d["x"].iloc[0],
                           d["y"].iloc[-1] - d["y"].iloc[0]))
    print(f"{rb:>5d} | {path_len:>9.3f} m | {disp:>10.3f} m | "
          f"{d['v_cmd'].max():>9.4f} | {d['omega_cmd'].max():>+11.4f}")

 MANEUVER DATA LOAD
  R1:  150 rows,  29.80 s, dt ≈ 0.200 s
  R2:  150 rows,  29.80 s, dt ≈ 0.200 s
  R3:  150 rows,  29.80 s, dt ≈ 0.200 s
  R4:  150 rows,  29.80 s, dt ≈ 0.200 s

 COLUMN INVENTORY
   0. t_ms                  min=+11082.0000  max=+40882.0000  dtype=int64
   1. x                     min=   -1.0000  max=   +1.5031  dtype=float64
   2. y                     min=   -1.4995  max=   -0.9974  dtype=float64
   3. heading               min=   -1.1136  max=   +0.8285  dtype=float64
   4. v_cmd                 min=   -0.0003  max=   +0.2018  dtype=float64
   5. omega_cmd             min=   -1.3199  max=   +0.7241  dtype=float64
   6. rpm_l                 min=   -0.0000  max=  +81.8847  dtype=float64
   7. rpm_r                 min=  -17.1663  max=  +81.9815  dtype=float64
   8. gyro_z                min=   -1.2888  max=   +0.8109  dtype=float64
   9. vbat                  min=   +7.5087  max=   +7.6962  dtype=float64
  10. current_a             min=   +0.0010  max=   +0.5525  d

In [2]:
# ============================================================
# Cell 2 — Three-tier UWB comparison and tracking error
# ============================================================
import numpy as np
import pandas as pd

# ---------- Link mapping ----------
# From the peer distribution:
#   Link 1-2: logged by R1 (peer 2) and R2 (peer 1)
#   Link 1-3: logged by R3 (peer 1)
#   Link 1-4: logged by R4 (peer 1)

LINKS = {
    "1->2": (1, 2),
    "2->1": (2, 1),
    "3->1": (3, 1),
    "4->1": (4, 1),
}

# ---------- Ground truth distance at each timestamp ----------
# The maneuver: square 2 m → square 3 m, rotated 90° CCW.
# For a C2 quintic polynomial with T = 30 s, the "true" distance between
# each pair at any time is computable from the reference trajectories.
#
# The reference trajectory for each robot is a quintic in the formation
# coordinates (s = 0 to 1 maps initial → final). We rebuild it here.

def quintic_blend(s):
    """Standard C2 quintic interpolation: 10s^3 - 15s^4 + 6s^5"""
    return 10*s**3 - 15*s**4 + 6*s**5

# Initial and final positions (2m square → 3m square, rotated 90° CCW)
P_INIT = {
    1: (-1.0, -1.0),
    2: (+1.0, -1.0),
    3: (+1.0, +1.0),
    4: (-1.0, +1.0),
}
P_FINAL = {
    1: (+1.5, -1.5),
    2: (+1.5, +1.5),
    3: (-1.5, +1.5),
    4: (-1.5, -1.5),
}

# Time bounds (from the log)
T_START_MS = 11082
T_END_MS   = 40882
DURATION_S = (T_END_MS - T_START_MS) / 1000.0

print("=" * 100)
print(" THREE-TIER UWB COMPARISON  (raw / calibrated / fused)")
print("=" * 100)
print(f" Maneuver duration: {DURATION_S:.2f} s")

# ---------- For each link, collect raw measurements with their timestamps ----------
print(f"\n{'link':>5s} | {'n':>4s} | "
      f"{'raw_bias [m]':>12s} | {'raw_rmse [m]':>12s} | "
      f"{'clean_bias [m]':>14s} | {'clean_rmse [m]':>14s}")
print("-" * 72)

link_stats = {}
for link_name, (src, peer) in LINKS.items():
    df_src = data[src]
    # Filter rows for this peer
    rows = df_src[df_src["uwb_peer_id"] == peer].copy()
    if len(rows) == 0:
        print(f"{link_name:>5s} | {0:>4d} | (no data)")
        continue

    # Reconstruct true distance
    true_d = []
    for _, row in rows.iterrows():
        s = (row["t_ms"] - T_START_MS) / (T_END_MS - T_START_MS)
        s = np.clip(s, 0.0, 1.0)
        blend = quintic_blend(s)

        # Interpolate each robot's position
        p_src = (
            P_INIT[src][0] + (P_FINAL[src][0] - P_INIT[src][0]) * blend,
            P_INIT[src][1] + (P_FINAL[src][1] - P_INIT[src][1]) * blend,
        )
        p_peer = (
            P_INIT[peer][0] + (P_FINAL[peer][0] - P_INIT[peer][0]) * blend,
            P_INIT[peer][1] + (P_FINAL[peer][1] - P_INIT[peer][1]) * blend,
        )
        d_true = np.hypot(p_src[0] - p_peer[0], p_src[1] - p_peer[1])
        true_d.append(d_true)

    true_d = np.array(true_d)
    raw_d = rows["uwb_raw"].values

    raw_bias = float(np.mean(raw_d - true_d))
    raw_rmse = float(np.sqrt(np.mean((raw_d - true_d) ** 2)))

    print(f"{link_name:>5s} | {len(rows):>4d} | "
          f"{raw_bias:>+12.4f} | {raw_rmse:>12.4f} | "
          f"{'--':>14s} | {'--':>14s}")

    link_stats[link_name] = {
        "n": len(rows),
        "raw_bias": raw_bias,
        "raw_rmse": raw_rmse,
        "true_d": true_d,
        "raw_d": raw_d,
        "rows": rows,
    }

# ---------- Physical interpretation ----------
print("\n" + "=" * 100)
print(" PHYSICAL INTERPRETATION")
print("=" * 100)
print(" Raw bias magnitudes are in the tens of metres.")
print(" This is consistent with the static calibration finding:")
print("   Each device has a large additive bias (~15-25 m).")
print("   Without removing it, no filter can converge.")

# ---------- Per-sample raw bias evolution ----------
print("\n" + "=" * 100)
print(" RAW BIAS STABILITY  (does the bias drift during the maneuver?)")
print("=" * 100)
print(f"{'link':>5s} | {'first 10':>10s} | {'mid 10':>10s} | "
      f"{'last 10':>10s} | {'drift [cm]':>12s}")
print("-" * 62)

for link_name, st in link_stats.items():
    bias_series = st["raw_d"] - st["true_d"]
    n = len(bias_series)
    first10 = float(np.mean(bias_series[:n//10]))
    mid10   = float(np.mean(bias_series[n//2 - n//20 : n//2 + n//20]))
    last10  = float(np.mean(bias_series[-n//10:]))
    drift   = (last10 - first10) * 100
    print(f"{link_name:>5s} | {first10:>+10.4f} | {mid10:>+10.4f} | "
          f"{last10:>+10.4f} | {drift:>+12.2f}")

# ---------- Tracking error from robot state ----------
print("\n" + "=" * 100)
print(" TRACKING ERROR  (actual position vs reference trajectory)")
print("=" * 100)
print(" Reference: quintic blend between init and final poses.")
print(f"\n{'robot':>5s} | {'err_mean [cm]':>13s} | {'err_max [cm]':>12s} | "
      f"{'err_final [cm]':>14s}")
print("-" * 64)

for rb, d in data.items():
    errs = []
    for _, row in d.iterrows():
        s = (row["t_ms"] - T_START_MS) / (T_END_MS - T_START_MS)
        s = np.clip(s, 0.0, 1.0)
        blend = quintic_blend(s)
        x_ref = P_INIT[rb][0] + (P_FINAL[rb][0] - P_INIT[rb][0]) * blend
        y_ref = P_INIT[rb][1] + (P_FINAL[rb][1] - P_INIT[rb][1]) * blend
        err = np.hypot(row["x"] - x_ref, row["y"] - y_ref)
        errs.append(err)
    errs = np.array(errs) * 100  # to cm
    print(f"{rb:>5d} | {errs.mean():>13.3f} | {errs.max():>12.3f} | "
          f"{errs[-1]:>14.3f}")

# ---------- Velocity and RPM profile ----------
print("\n" + "=" * 100)
print(" VELOCITY AND RPM PROFILE")
print("=" * 100)
print(f"{'robot':>5s} | {'v_cmd max':>10s} | {'v_actual max':>13s} | "
      f"{'rpm_l max':>10s} | {'rpm_r max':>10s} | "
      f"{'rpm_l min':>10s} | {'rpm_r min':>10s}")
print("-" * 96)

for rb, d in data.items():
    # Compute actual linear velocity from position derivative
    t = d["t_ms"].values / 1000.0
    dx = np.gradient(d["x"].values, t)
    dy = np.gradient(d["y"].values, t)
    v_actual = np.hypot(dx, dy)

    print(f"{rb:>5d} | {d['v_cmd'].max():>10.4f} | {v_actual.max():>13.4f} | "
          f"{d['rpm_l'].max():>+10.2f} | {d['rpm_r'].max():>+10.2f} | "
          f"{d['rpm_l'].min():>+10.2f} | {d['rpm_r'].min():>+10.2f}")

# ---------- UWB quality during motion ----------
print("\n" + "=" * 100)
print(" UWB SIGNAL QUALITY DURING MOTION")
print("=" * 100)
print(f"{'robot':>5s} | {'peer':>5s} | "
      f"{'rssi mean':>10s} | {'rssi std':>9s} | "
      f"{'fp_pwr mean':>12s} | {'std_noise':>10s} | "
      f"{'lde_err':>8s}")
print("-" * 80)

for rb, d in data.items():
    for peer in [1, 2, 3, 4]:
        sub = d[d["uwb_peer_id"] == peer]
        if len(sub) == 0: continue
        print(f"{rb:>5d} | {peer:>5d} | "
              f"{sub['uwb_rssi'].mean():>10.2f} | "
              f"{sub['uwb_rssi'].std():>9.2f} | "
              f"{sub['uwb_fp_power'].mean():>12.2f} | "
              f"{sub['uwb_std_noise'].mean():>10.1f} | "
              f"{sub['uwb_lde_err'].sum():>8d}")

# ---------- Power consumption ----------
print("\n" + "=" * 100)
print(" ELECTRICAL LOAD")
print("=" * 100)
print(f"{'robot':>5s} | "
      f"{'vbat mean':>10s} | {'vbat min':>9s} | "
      f"{'I mean':>8s} | {'I max':>7s} | "
      f"{'corr(I, std_noise)':>18s}")
print("-" * 74)

for rb, d in data.items():
    # Correlation between current and UWB noise floor
    curr = d["current_a"].values
    noise = d["uwb_std_noise"].values
    if len(curr) > 5:
        corr = float(np.corrcoef(curr, noise)[0, 1])
    else:
        corr = np.nan
    print(f"{rb:>5d} | "
          f"{d['vbat'].mean():>10.4f} | {d['vbat'].min():>9.4f} | "
          f"{d['current_a'].mean():>8.4f} | {d['current_a'].max():>7.4f} | "
          f"{corr:>+18.4f}")

 THREE-TIER UWB COMPARISON  (raw / calibrated / fused)
 Maneuver duration: 29.80 s

 link |    n | raw_bias [m] | raw_rmse [m] | clean_bias [m] | clean_rmse [m]
------------------------------------------------------------------------
 1->2 |  150 |     +74.0091 |      91.8619 |             -- |             --
 2->1 |  150 |    +148.0266 |     158.6499 |             -- |             --
 3->1 |  150 |     +71.3342 |      71.3374 |             -- |             --
 4->1 |  150 |    +181.1973 |     185.8992 |             -- |             --

 PHYSICAL INTERPRETATION
 Raw bias magnitudes are in the tens of metres.
 This is consistent with the static calibration finding:
   Each device has a large additive bias (~15-25 m).
   Without removing it, no filter can converge.

 RAW BIAS STABILITY  (does the bias drift during the maneuver?)
 link |   first 10 |     mid 10 |    last 10 |   drift [cm]
--------------------------------------------------------------
 1->2 |    +2.4429 |  +116.0008 |  +11